# Rescore: Zero-Shot & Few-Shot Extraction (SROIE)
**Group 1 – Invoices | Utility notebook**

This notebook rescores zero-shot and few-shot extraction predictions stored as `*_results.json` files.
It reconstructs SROIE gold labels from the HuggingFace Parquet mirror (no loading script required),
then applies reinforced normalisers for `total` (currency removal) and `date` (ISO canonicalisation)
to report Exact Match, Token F1, and Char F1 per field and overall.

**Cells:**
- Cell 1 — `build_golds`: reconstruct gold labels from HF Parquet
- Cell 2 — Locate result JSON files in Kaggle working/input directories
- Cell 3 — Copy zero-shot result files into the working directory
- Cell 4 — `rescore` script: normalise predictions and compute all metrics
- Cell 5 — Few-shot rescoring: locate and copy few-shot result files
- Cell 6 — Copy both zero-shot and few-shot results with distinguishing suffixes
- Cell 7 — Run the rescorer over all copied files
- Cell 8 — Debug: inspect SmolVLM few-shot predictions


## 0 — Environment setup (Colab / Kaggle / Local)


In [ ]:
import os


In [ ]:
def _is_kaggle():
    return os.path.exists('/kaggle/working')
def _is_colab():
    try:
        import google.colab
        return not _is_kaggle()
    except ImportError:
        return False
PLATFORM = 'kaggle' if _is_kaggle() else ('colab' if _is_colab() else 'local')
print(PLATFORM)


In [ ]:
if PLATFORM == 'colab':
    USE_DRIVE = True  # Set False to save in /content/outputs without mounting Drive
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/NLP_Invoices/outputs'
    else:
        OUTPUT_DIR = '/content/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    from huggingface_hub import get_token
    HF_TOKEN = get_token()
    if not HF_TOKEN:
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get('HF_TOKEN')
        except Exception: pass
    if not HF_TOKEN: raise ValueError('HF_TOKEN not found.')
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('✅ Colab ready')


In [ ]:
if PLATFORM == 'kaggle':
    OUTPUT_DIR = '/kaggle/working/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    HF_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN loaded from Kaggle Secrets')
    except Exception as _e:
        print(f'Kaggle Secrets not available: {_e}')
    if not HF_TOKEN: raise ValueError('HF_TOKEN not found.')
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print(f'✅ Kaggle ready. Output → {OUTPUT_DIR}')


In [ ]:
if PLATFORM == 'local':
    OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    from huggingface_hub import get_token, login
    HF_TOKEN = get_token() or os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN: raise ValueError('HF_TOKEN not found.')
    login(token=HF_TOKEN)
    print(f'✅ Local ready. Output → {OUTPUT_DIR}')


In [9]:
#!/usr/bin/env python3
"""
build_golds.py
==============

Rebuilds SROIE gold labels on `datasets` >= 4.x, where the loading-script
`sroie.py` from `darentang/sroie` is NO longer supported
(RuntimeError: Dataset scripts are no longer supported).

Strategy (tries in order, uses the first that works):
  1. Reads the auto-converted Parquet files from HF via `hf_hub` (refs/convert/parquet).
     No loading script required -> works with datasets 4.x.
  2. If the Parquet already exposes the fields (company/date/address/total) uses them directly.
  3. If it exposes words + ner_tags instead, rebuilds the gold labels as in the main notebook.

Output: saves /kaggle/working/golds.json (list of 347 dicts, test split).

Usage (Kaggle, in a cell):
    !python build_golds.py
or paste the content into a cell and call  build_and_save()
"""

import os
import re
import json

FIELDS = ["company", "date", "address", "total"]
REPO = "darentang/sroie"
OUT = "/kaggle/working/golds.json"


def reconstruct_fields(words, ner_tags) -> dict:
    """Identical to the notebook logic: reconstructs gold labels from words + ner_tags."""
    TAG2FIELD = {1: "company", 2: "company",
                 3: "date",    4: "date",
                 5: "address", 6: "address",
                 7: "total",   8: "total"}
    spans = {f: [] for f in FIELDS}
    current_field = None
    for word, tag in zip(words, ner_tags):
        field = TAG2FIELD.get(int(tag))
        is_begin = int(tag) in (1, 3, 5, 7)
        if field is None:
            current_field = None
            continue
        clean = re.sub(r'[^a-zA-Z0-9]', '', str(word))
        if not clean:
            continue
        if is_begin and not spans[field]:
            current_field = field
        if field == current_field:
            spans[field].append(str(word))
    return {f: " ".join(spans[f]) for f in FIELDS}


def _golds_from_dataframe(df) -> list:
    """Extracts gold labels from a pandas DataFrame regardless of its schema."""
    cols = set(df.columns)
    golds = []

    # Case A: the 4 fields are direct columns
    if {"company", "date", "address", "total"}.issubset(cols):
        for _, row in df.iterrows():
            golds.append({f: ("" if row[f] is None else str(row[f])) for f in FIELDS})
        return golds

    # Case B: fields nested in an 'entities' or 'key' struct
    for nest in ("entities", "key"):
        if nest in cols:
            for _, row in df.iterrows():
                ent = row[nest]
                if isinstance(ent, dict):
                    golds.append({f: str(ent.get(f, "") or "") for f in FIELDS})
                else:
                    golds.append({f: "" for f in FIELDS})
            return golds

    # Case C: reconstruct from words + ner_tags
    if {"words", "ner_tags"}.issubset(cols):
        for _, row in df.iterrows():
            golds.append(reconstruct_fields(row["words"], row["ner_tags"]))
        return golds

    raise RuntimeError(f"Unexpected Parquet schema. Available columns: {sorted(cols)}")


def build_golds() -> list:
    """Downloads the auto-converted Parquet and rebuilds gold labels for the test split."""
    import pandas as pd
    from huggingface_hub import hf_hub_download, list_repo_files

    # Auto-converted Parquet files live in the 'refs/convert/parquet' revision
    files = list_repo_files(REPO, repo_type="dataset", revision="refs/convert/parquet")
    parquet_files = [f for f in files if f.endswith(".parquet")]
    # Keep only the test split
    test_files = [f for f in parquet_files if "test" in f.lower()]
    if not test_files:
        # Fallback: some repos name files differently
        test_files = [f for f in parquet_files if "/test" in f or f.startswith("test")]
    if not test_files:
        raise RuntimeError(f"No test Parquet found. Available files: {parquet_files}")

    print(f"Test Parquet files found: {test_files}")
    dfs = []
    for pf in sorted(test_files):
        local = hf_hub_download(REPO, pf, repo_type="dataset",
                                revision="refs/convert/parquet")
        dfs.append(pd.read_parquet(local))
    df = pd.concat(dfs, ignore_index=True)
    print(f"Test rows: {len(df)} | columns: {list(df.columns)}")

    golds = _golds_from_dataframe(df)
    print(f"Gold labels reconstructed: {len(golds)}")
    return golds


def build_and_save(out_path: str = OUT) -> list:
    golds = build_golds()
    with open(out_path, "w") as f:
        json.dump(golds, f, ensure_ascii=False)
    print(f"Saved: {out_path} ({len(golds)} examples)")
    # Show 3 examples for sanity check
    for i in range(min(3, len(golds))):
        print(f"  [{i}] {golds[i]}")
    return golds


if __name__ == "__main__":
    build_and_save()


Parquet test trovati: ['sroie/test/0000.parquet']
Righe test: 347 | colonne: ['id', 'words', 'bboxes', 'ner_tags', 'image_path']
Gold ricostruiti: 347
Salvato: /kaggle/working/golds.json (347 esempi)
  [0] {'company': 'OJC MARKETING SDN BHD', 'date': '15/01/2019', 'address': 'NO 2 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR', 'total': '193.00'}
  [1] {'company': 'OJC MARKETING SDN BHD', 'date': '02/01/2019', 'address': 'NO 2 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR', 'total': '170.00'}
  [2] {'company': 'PERNIAGAAN ZHENG HUI', 'date': '09/02/2018', 'address': 'NO.59 JALAN PERMAS 9/5 BANDAR BARU PERMAS JAYA 81750 JOHOR BAHRU', 'total': '436.20'}


In [14]:
import glob
# Search everywhere in inputs and working directory
print("In working:", glob.glob("/kaggle/working/*results*.json"))
print("In inputs:", glob.glob("/kaggle/input/**/*results*.json", recursive=True))

In working: []
Negli input: ['/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/smol_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/pali_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/qwen_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/baseline_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/qwen2b_results.json']


In [15]:
import shutil, glob, os
src = "/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie"
for f in glob.glob(os.path.join(src, "*_results.json")):
    shutil.copy(f, "/kaggle/working/")
    print("copied:", os.path.basename(f))
print("\nNow in working:", [os.path.basename(x) for x in glob.glob("/kaggle/working/*_results.json")])

copiato: smol_results.json
copiato: pali_results.json
copiato: qwen_results.json
copiato: baseline_results.json
copiato: qwen2b_results.json

Ora in working: ['qwen2b_results.json', 'pali_results.json', 'baseline_results.json', 'smol_results.json', 'qwen_results.json']


In [16]:
#!/usr/bin/env python3

import os
import re
import glob
import json
import argparse
from collections import Counter
from datetime import datetime

FIELDS = ["company", "date", "address", "total"]


# ---------------------------------------------------------------------------
# 1. GOLD RECONSTRUCTION  (identical to main notebook)
# ---------------------------------------------------------------------------
def reconstruct_fields(example: dict) -> dict:
    TAG2FIELD = {1: "company", 2: "company",
                 3: "date",    4: "date",
                 5: "address", 6: "address",
                 7: "total",   8: "total"}
    spans = {f: [] for f in FIELDS}
    current_field = None
    for word, tag in zip(example["words"], example["ner_tags"]):
        field = TAG2FIELD.get(tag)
        is_begin = tag in (1, 3, 5, 7)
        if field is None:
            current_field = None
            continue
        clean = re.sub(r'[^a-zA-Z0-9]', '', word)
        if not clean:
            continue
        if is_begin and not spans[field]:
            current_field = field
        if field == current_field:
            spans[field].append(word)
    return {f: " ".join(spans[f]) for f in FIELDS}


def load_golds(gold_json: str | None):
    """Load gold labels from a JSON file if provided, otherwise from the SROIE dataset."""
    if gold_json:
        with open(gold_json) as f:
            golds = json.load(f)
        print(f"Gold labels loaded from {gold_json}: {len(golds)} examples")
        return golds
    from datasets import load_dataset
    print("Loading SROIE dataset (darentang/sroie, split=test)...")
    ds = load_dataset("darentang/sroie", split="test", trust_remote_code=True)
    golds = [reconstruct_fields(ex) for ex in ds.remove_columns(["image_path"])]
    print(f"Gold labels reconstructed: {len(golds)} examples")
    return golds


# ---------------------------------------------------------------------------
# 2. NORMALISERS
# ---------------------------------------------------------------------------
def normalise(text) -> str:
    """Generic normalisation for text fields (company, address)."""
    if text is None or not isinstance(text, str):
        return ""
    text = re.sub(r'(?i)^\s*(rm|myr|\$|usd)\s*', '', text)
    text = re.sub(r'(\d),(\d)', r'\1.\2', text)
    text = re.sub(r'[,\.]', ' ', text)
    return " ".join(text.lower().strip().split())


def normalise_amount(text: str) -> str:
    """Normalise the `total` field. Reinforced version:
    removes currency symbols anywhere in the string (not just at the start) and supports
    EUR/GBP. Output: float with 2 decimal places."""
    if text is None or not isinstance(text, str):
        return ""
    # Remove currency prefixes/suffixes anywhere in the string
    text = re.sub(r'(?i)\b(rm|myr|usd|sgd|us)\b', ' ', text)
    text = re.sub(r'[$€£]', ' ', text)
    text = text.strip()
    # Thousands separator: 1,234.50 -> 1234.50
    text = re.sub(r'(\d),(\d{3})(?=[.\s]|$)', r'\1\2', text)
    # Decimal comma -> dot: 12,50 -> 12.50
    text = re.sub(r'(\d),(\d{1,2})(?=\s|$)', r'\1.\2', text)
    try:
        amount = float(re.sub(r'[^\d.]', '', text))
        return f"{amount:.2f}"
    except (ValueError, TypeError):
        return normalise(text)


def normalise_amount_naive(text: str) -> str:
    """Version WITHOUT currency removal: used only to measure its impact.
    Applies lowercasing and whitespace collapse but keeps currency symbols."""
    if text is None or not isinstance(text, str):
        return ""
    return " ".join(text.lower().strip().split())


# --- DATE normalisation ----------------------------------------------------
_MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun",
     "jul", "aug", "sep", "oct", "nov", "dec"], start=1)}

_DATE_FORMATS = [
    "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y", "%d-%m-%y",
    "%Y-%m-%d", "%Y/%m/%d", "%Y%m%d",
    "%m/%d/%Y", "%m-%d-%Y", "%m/%d/%y", "%m-%d-%y",
    "%d %b %Y", "%d %B %Y", "%d %b %y",
]


def normalise_date(text: str) -> str:
    """Normalise a date string to canonical ISO format 'YYYY-MM-DD' when possible.

    NOTE: day/month ambiguity (e.g. 12/01/19) is unresolvable without context.
    We assume the dominant SROIE format: day-first (DD/MM).
    Formats with textual months (05 MAR 2018) and ISO (20180304) are unambiguous.
    Falls back to generic text normalisation if no parsing succeeds.
    """
    if text is None or not isinstance(text, str) or not text.strip():
        return ""
    t = text.strip().lower()

    # Textual month: "05 mar 2018" / "28 mar 18"
    m = re.match(r'(\d{1,2})\s+([a-z]{3,})\s+(\d{2,4})', t)
    if m:
        d, mon, y = m.group(1), m.group(2)[:3], m.group(3)
        if mon in _MONTHS:
            y = int(y); y += 2000 if y < 100 else 0
            try:
                return f"{y:04d}-{_MONTHS[mon]:02d}-{int(d):02d}"
            except ValueError:
                pass

    # Known numeric date formats
    cleaned = re.sub(r'\s+', '', text.strip())
    for fmt in _DATE_FORMATS:
        try:
            dt = datetime.strptime(cleaned, fmt)
            # 2-digit year heuristic already handled by strptime (%y -> 2000+)
            return dt.strftime("%Y-%m-%d")
        except ValueError:
            continue
    # Fallback: generic text normalisation
    return normalise(text)


# ---------------------------------------------------------------------------
# 3. METRICS
# ---------------------------------------------------------------------------
def _f1(pred_units, gold_units) -> float:
    if not pred_units or not gold_units:
        return float(pred_units == gold_units)
    common = Counter(pred_units) & Counter(gold_units)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p = n / len(pred_units)
    r = n / len(gold_units)
    return 2 * p * r / (p + r)


def score_field(preds, golds, field, total_norm, date_norm):
    """Returns mean (em, tf1, cf1) for a field, given the chosen normalisers."""
    def norm(value):
        if field == "total":
            return total_norm(value)
        if field == "date":
            return date_norm(value)
        return normalise(value)

    em, tf1, cf1 = [], [], []
    for pred, gold in zip(preds, golds):
        p = norm(pred.get(field, ""))
        g = norm(gold.get(field, ""))
        em.append(int(p == g))
        tf1.append(_f1(p.split(), g.split()))
        cf1.append(_f1(list(p), list(g)))
    n = len(em)
    return (sum(em) / n, sum(tf1) / n, sum(cf1) / n)


# ---------------------------------------------------------------------------
# 4. MAIN
# ---------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results-dir", default=".",
                    help="directory containing *_results.json files (default: current)")
    ap.add_argument("--gold-json", default=None,
                    help="JSON file with gold labels (avoids downloading the dataset)")
    args = ap.parse_args()

    golds = load_golds(args.gold_json)

    files = sorted(glob.glob(os.path.join(args.results_dir, "*_results.json")))
    if not files:
        print(f"\nNo *_results.json files found in {args.results_dir!r}.")
        print("Pass --results-dir with the correct path (e.g. /kaggle/working).")
        return

    print(f"\nFound {len(files)} result files:")
    for f in files:
        print("  -", os.path.basename(f))

    summary = {}   # model -> dict of metrics

    for path in files:
        with open(path) as f:
            data = json.load(f)
        preds = data.get("predictions")
        if not preds:
            print(f"\n[skip] {os.path.basename(path)}: nessuna chiave 'predictions'.")
            continue
        if len(preds) != len(golds):
            print(f"\n[warn] {os.path.basename(path)}: {len(preds)} preds vs "
                  f"{len(golds)} gold — confronto troncato al minimo.")
        n = min(len(preds), len(golds))
        p, g = preds[:n], golds[:n]

        model = os.path.basename(path).replace("_results.json", "")

        # --- total: with vs without currency normalisation ---
        em_total_norm, _, _ = score_field(p, g, "total",
                                           normalise_amount, normalise_date)
        em_total_naive, _, _ = score_field(p, g, "total",
                                            normalise_amount_naive, normalise_date)

        # --- date: with vs without format normalisation ---
        em_date_norm, _, _ = score_field(p, g, "date",
                                          normalise_amount, normalise_date)
        em_date_textonly, _, _ = score_field(p, g, "date",
                                              normalise_amount, normalise)

        # --- full metrics with reinforced normalisation ---
        full = {}
        for fld in FIELDS:
            em, tf1, cf1 = score_field(p, g, fld, normalise_amount, normalise_date)
            full[f"em_{fld}"] = round(em, 3)
            full[f"tf1_{fld}"] = round(tf1, 3)
            full[f"cf1_{fld}"] = round(cf1, 3)
        full["em_overall"] = round(sum(full[f"em_{f}"] for f in FIELDS) / 4, 3)
        full["tf1_overall"] = round(sum(full[f"tf1_{f}"] for f in FIELDS) / 4, 3)
        full["cf1_overall"] = round(sum(full[f"cf1_{f}"] for f in FIELDS) / 4, 3)

        summary[model] = full

        print(f"\n{'='*60}")
        print(f"  {model}")
        print(f"{'='*60}")
        print(f"  IMPACT OF CURRENCY NORMALISATION (total field, EM):")
        print(f"    without currency removal : {em_total_naive:.3f}")
        print(f"    with  currency removal   : {em_total_norm:.3f}   "
              f"(+{em_total_norm - em_total_naive:.3f})")
        print(f"  IMPACT OF DATE NORMALISATION (date field, EM):")
        print(f"    text-only               : {em_date_textonly:.3f}")
        print(f"    canonical ISO format     : {em_date_norm:.3f}   "
              f"(+{em_date_norm - em_date_textonly:.3f})")
        print(f"  FULL METRICS (reinforced normalisation):")
        print(f"  {'Field':<10} {'EM':>7} {'TokenF1':>9} {'CharF1':>8}")
        print(f"  {'-'*38}")
        for fld in FIELDS:
            print(f"  {fld:<10} {full[f'em_{fld}']:>7.3f} "
                  f"{full[f'tf1_{fld}']:>9.3f} {full[f'cf1_{fld}']:>8.3f}")
        print(f"  {'-'*38}")
        print(f"  {'OVERALL':<10} {full['em_overall']:>7.3f} "
              f"{full['tf1_overall']:>9.3f} {full['cf1_overall']:>8.3f}")

    # --- final comparison table ---
    if summary:
        print(f"\n\n{'#'*60}")
        print("  FINAL COMPARISON (EM overall, reinforced normalisation)")
        print(f"{'#'*60}")
        print(f"  {'Model':<22} {'EM':>7} {'TokenF1':>9} {'CharF1':>8}")
        print(f"  {'-'*48}")
        for model, m in sorted(summary.items(),
                               key=lambda kv: kv[1]["em_overall"], reverse=True):
            print(f"  {model:<22} {m['em_overall']:>7.3f} "
                  f"{m['tf1_overall']:>9.3f} {m['cf1_overall']:>8.3f}")

        out = os.path.join(args.results_dir, "rescored_summary.json")
        with open(out, "w") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        print(f"\nSaved: {out}")


import sys
sys.argv = ["rescore", "--results-dir", "/kaggle/working", "--gold-json", "/kaggle/working/golds.json"]
main()

Gold caricati da /kaggle/working/golds.json: 347 esempi

Trovati 5 file di risultati:
  - baseline_results.json
  - pali_results.json
  - qwen2b_results.json
  - qwen_results.json
  - smol_results.json

  baseline
  IMPATTO NORMALIZZAZIONE VALUTA (campo total, EM):
    senza rimozione valuta : 0.205
    con  rimozione valuta  : 0.288   (+0.084)
  IMPATTO NORMALIZZAZIONE DATA (campo date, EM):
    solo testo             : 0.700
    formato canonico ISO   : 0.706   (+0.006)
  METRICHE COMPLETE (normalizzazione rinforzata):
  Field           EM   TokenF1   CharF1
  --------------------------------------
  company      0.112     0.272    0.454
  date         0.706     0.706    0.816
  address      0.063     0.500    0.631
  total        0.288     0.288    0.622
  --------------------------------------
  OVERALL      0.292     0.442    0.631

  pali
  IMPATTO NORMALIZZAZIONE VALUTA (campo total, EM):
    senza rimozione valuta : 0.565
    con  rimozione valuta  : 0.648   (+0.084)
  IMPATTO 

In [17]:
# Few-shot rescoring — find result files

import glob
print(glob.glob("/kaggle/input/**/*_results.json", recursive=True))

['/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/smol_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/pali_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/qwen_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/baseline_results.json', '/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie/qwen2b_results.json', '/kaggle/input/notebooks/robertostoian/few-shot-information-extraction-sroie/smol_results.json', '/kaggle/input/notebooks/robertostoian/few-shot-information-extraction-sroie/pali_results.json', '/kaggle/input/notebooks/robertostoian/few-shot-information-extraction-sroie/qwen_results.json', '/kaggle/input/notebooks/robertostoian/few-shot-information-extraction-sroie/baseline_results.json', '/kaggle/input/notebooks/robertostoian/few-shot-information-extraction-sroie/qwen2b_results.json']


In [18]:
import shutil, glob, os

ZERO = "/kaggle/input/notebooks/robertostoian/zero-shot-information-extraction-sroie"
FEW  = "/kaggle/input/notebooks/robertostoian/few-shot-information-extraction-sroie"

# Clean up any stale files in working to avoid duplicates
for f in glob.glob("/kaggle/working/*_results.json"):
    os.remove(f)

# Copy zero-shot results with _zeroshot suffix
for f in glob.glob(os.path.join(ZERO, "*_results.json")):
    base = os.path.basename(f).replace("_results.json", "_zeroshot_results.json")
    shutil.copy(f, os.path.join("/kaggle/working", base))

# Copy few-shot results with _fewshot suffix
for f in glob.glob(os.path.join(FEW, "*_results.json")):
    base = os.path.basename(f).replace("_results.json", "_fewshot_results.json")
    shutil.copy(f, os.path.join("/kaggle/working", base))

print("Files in working:")
for f in sorted(glob.glob("/kaggle/working/*_results.json")):
    print("  ", os.path.basename(f))

File in working:
   baseline_fewshot_results.json
   baseline_zeroshot_results.json
   pali_fewshot_results.json
   pali_zeroshot_results.json
   qwen2b_fewshot_results.json
   qwen2b_zeroshot_results.json
   qwen_fewshot_results.json
   qwen_zeroshot_results.json
   smol_fewshot_results.json
   smol_zeroshot_results.json


In [19]:
import sys
sys.argv = ["rescore", "--results-dir", "/kaggle/working", "--gold-json", "/kaggle/working/golds.json"]
main()

Gold caricati da /kaggle/working/golds.json: 347 esempi

Trovati 10 file di risultati:
  - baseline_fewshot_results.json
  - baseline_zeroshot_results.json
  - pali_fewshot_results.json
  - pali_zeroshot_results.json
  - qwen2b_fewshot_results.json
  - qwen2b_zeroshot_results.json
  - qwen_fewshot_results.json
  - qwen_zeroshot_results.json
  - smol_fewshot_results.json
  - smol_zeroshot_results.json

  baseline_fewshot
  IMPATTO NORMALIZZAZIONE VALUTA (campo total, EM):
    senza rimozione valuta : 0.205
    con  rimozione valuta  : 0.288   (+0.084)
  IMPATTO NORMALIZZAZIONE DATA (campo date, EM):
    solo testo             : 0.700
    formato canonico ISO   : 0.706   (+0.006)
  METRICHE COMPLETE (normalizzazione rinforzata):
  Field           EM   TokenF1   CharF1
  --------------------------------------
  company      0.112     0.272    0.454
  date         0.706     0.706    0.816
  address      0.063     0.500    0.631
  total        0.288     0.288    0.622
  --------------------

In [20]:
import json
with open("/kaggle/working/smol_fewshot_results.json") as f:
    d = json.load(f)
preds = d.get("predictions", [])
print("Number of predictions:", len(preds))
print("First 5:", preds[:5])
# Count completely empty predictions
empty = sum(1 for p in preds if not any(p.get(k,"") for k in ["company","date","address","total"]))
print(f"Completely empty predictions: {empty}/{len(preds)}")

Numero predizioni: 347
Prime 5: [{'company': '', 'date': '', 'address': '', 'total': ''}, {'company': '', 'date': '', 'address': '', 'total': ''}, {'company': '', 'date': '', 'address': '', 'total': ''}, {'company': '', 'date': '', 'address': '', 'total': ''}, {'company': '', 'date': '', 'address': '', 'total': ''}]
Predizioni totalmente vuote: 347/347
